# Investigate the Payment Failures in Grafana

**Module 3: Visualizing Observability Data** · [Lesson](course.md) · [Quiz](quiz03_observability_visualization.ipynb)

Since Module 1, about half of the store's orders fail. In this Lab you investigate the failure in Grafana, as an on-call engineer would: you search the logs for errors, find the failed checkouts in the storefront's access log, open the trace of one of them, and find the error where the failure starts. You check that the host's CPU is not the cause, and you build a dashboard that shows the incident at a glance. After each step in Grafana, a notebook cell checks the same thing with SQL, which you can also run in any MySQL client.

Plan about 45 minutes. You need the DOG stack and the demo from Module 1, with the payment failure turned on (section 5 of Lab 1). Most steps take place in Grafana, in your browser; run the cells around them in order. The steps describe Grafana 11.4.0 and its Doris App plugin 1.0.5, which the course installs.

### Initialize the Lab

Run this cell once before using `lab.sql(...)` or the other helpers.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

## 1. Check that orders are failing

The investigation needs failed orders from the last few minutes. Count the orders of the last 15 minutes, and how many failed:

In [ ]:
orders = lab.wait_for_rows(
    """
    SELECT COUNT(*) AS orders,
           SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_orders,
           MAX(timestamp) AS latest_order
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
    HAVING SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) > 0
    """,
    title="Orders and failed orders of the last 15 minutes",
    hint="No order failed in the last 15 minutes. Check that the demo runs (lab.start_demo() in Module 1) and "
         "that paymentFailure is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)

**Expected result**

One row: the load generator's orders of the last 15 minutes, usually 30 to 60, about half of them failed, and the time of the latest order in UTC. If you turned on the flag less than 15 minutes ago, fewer orders have failed.

## 2. Open Discover and set the time range

The Doris App is a Grafana plugin for searching the telemetry in Doris. Its **Discover** page searches the rows of one table. Open [http://127.0.0.1:23000/a/velodb-doris-app/discover](http://127.0.0.1:23000/a/velodb-doris-app/discover) (or the port in `DORIS_OBS_GRAFANA_PORT`). You do not need to sign in to search: Grafana lets visitors view its pages.

The bar at the top of the page says what to search:

| Field | What it does | In this Lab |
| --- | --- | --- |
| **Datasource** | The Grafana data source that connects to Doris | `Doris` |
| **Database**, **Table** | The table to search | `otel`, `otel_logs` |
| **Mode** | How you write a search: **Lucene**, a short `field:value` syntax, or **SQL**, a `WHERE` condition | `Lucene` |
| Search box | The search itself; the box's hint shows the syntax of the mode | Empty for now |
| **Time Field** | The column that the time range applies to | `timestamp` |
| **Time Range** | Which rows to search, by time | The two values that the next cell prints |
| **Query** | Runs the search | |

The page remembers these settings from your last visit, so check them each time you open it.

Doris stores the telemetry's times in UTC, and so do the rows that the page shows. The Doris App, however, compares its time range with those times as if they were in your computer's time zone. Unless your computer uses UTC, a relative range such as **Last 15 minutes** therefore searches the wrong period and misses the newest rows, or finds none. Set an absolute range in UTC instead. The next cell prints one that starts 30 minutes ago and ends 30 minutes from now, so that it also covers the rows that arrive while you work:

In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
TIME_FROM = str(now - timedelta(minutes=30))
TIME_TO = str(now + timedelta(minutes=30))
print("From:", TIME_FROM)
print("To:  ", TIME_TO)

**Expected result**

Two times in UTC, an hour apart, such as `From: 2026-10-20 19:35:12` and `To: 2026-10-20 20:35:12`. The SQL cells below use the same range, so their numbers match what the page shows.

In Discover:

1. Click the **Time Range** button (it shows **Last 1 day** on your first visit). In **Absolute time range**, replace the **From** value (`now-1d`) with the From time above and the **To** value (`now`) with the To time, and click **Apply time range**.
2. Check that **Table** is `otel_logs` and **Mode** is **Lucene**, and that the search box is empty.
3. Click **Query**.

The page counts the matching rows (**hits**), draws them per time interval, and lists the newest first, each with its columns. Every `trace_id` in the list is a link. When you work past the To time, run the previous cell again and set the new range.

## 3. Look for error logs

The first search of an investigation is often for errors. `severity_text` is the label that each source writes, such as `ERROR` or `Error`, while `severity_number` puts every source on one OpenTelemetry scale, on which 17 to 20 is ERROR and 21 to 24 FATAL. In the search box, type:

```text
severity_number:>=17
```

and click **Query**. The notebook runs the same search with SQL, counted by service:

In [ ]:
lab.sql(f"""
SELECT service_name, COUNT(*) AS error_logs, MAX(timestamp) AS latest_error
FROM otel_logs
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND severity_number >= 17
GROUP BY service_name
ORDER BY error_logs DESC
""", title="Error logs in the time range, by service");

**Expected result**

Discover shows a few hits at most, and the cell the same counts: errors that the Doris FE logged, which Module 2 loaded, and perhaps one `Consume error` that `accounting` logged when the demo started. If the FE logged its startup errors before the From time, there are no hits at all. None of them is about payments. The failed orders do not appear as error logs, because the payment and checkout services send no logs to a Collector and the access log of the storefront's proxy has no severity (Module 2). A search for errors by severity misses this incident, so search the access log instead.

## 4. Find the failed checkouts in the access log

The storefront's proxy writes one access log line for each request, with the method, the path, and the HTTP status code, such as `"POST /api/checkout HTTP/1.1" 500`. A failed order is a checkout request that ended with status 500.

In Lucene mode, a quoted text after `body:` finds rows whose `body` contains those words next to each other, in that order. Search for the checkout requests that ended with status 500:

```text
body:"POST /api/checkout" AND body:"HTTP/1.1 500"
```

`AND` keeps the rows that match both phrases. One phrase, `"POST /api/checkout HTTP/1.1 500"`, would find the load generator's orders but miss the ones that you place in the storefront page yourself: the page adds a query string to the path, `/api/checkout?currencyCode=USD`, and its words would stand between `checkout` and `HTTP`.

Then switch **Mode** to **SQL**. The search box now takes a SQL `WHERE` condition. The Lucene search above becomes this condition:

```text
body MATCH_PHRASE 'POST /api/checkout' AND body MATCH_PHRASE 'HTTP/1.1 500'
```

Type it in the search box and click **Query** again: the hits are the same.

In [ ]:
lab.sql(f"""
SELECT COUNT(*) AS failed_checkouts, MIN(timestamp) AS first_failure, MAX(timestamp) AS latest_failure
FROM otel_logs
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND body MATCH_PHRASE 'POST /api/checkout'
  AND body MATCH_PHRASE 'HTTP/1.1 500'
""", title="Failed checkouts in the access log");

**Expected result**

The number of hits in Discover: one for each order that failed since you turned on the flag, or since the From time if the flag has been on longer, about one a minute. `first_failure` is close to the time when you saved the flag, unless that was before the From time. The same orders failed in Module 1's count of checkout spans with an error status: the access log and the traces count the same failures.

`MATCH_PHRASE` does not scan the text of every row. The Doris exporter created `otel_logs` with an inverted index on `body`, which splits each body into words and records the rows that contain each word. Doris looks up the rows that contain the words of a phrase, then keeps those where the words are adjacent and in the phrase's order. See how the index splits the start of a line that the storefront page sent:

In [ ]:
lab.sql("""
SELECT TOKENIZE('"POST /api/checkout?currencyCode=USD HTTP/1.1" 500 - via_upstream',
                '"parser" = "unicode", "lower_case" = "true"') AS tokens
""", title="The words that the index of otel_logs.body makes of an access log line");

**Expected result**

A list of words: `post`, `api`, `checkout`, `currencycode`, `usd`, `http`, `1.1`, `500`, and `via_upstream`. The index lowercases the words and drops the quotes, slashes, question marks, and dashes, so a phrase does not need them and matches `POST` in any case. The words `currencycode` and `usd` are what would break the single long phrase. Module 6 measures how much reading the index saves.

## 5. Open the trace of a failed checkout

Each access log line carries the trace ID of its request. In the list of hits, click the `trace_id` of a row that is at least a minute old: the services export their spans in batches, so the spans of the newest request can still be on their way to Doris. The **Trace Panel** opens on the right with the request's trace as a waterfall.

<img src="images/discover-failed-checkouts.png" alt="The Discover page of the Doris App: the table otel_logs, the Lucene search for failed checkouts, an absolute time range, the hits per interval, and access log lines with HTTP 500 and their trace IDs" style="max-width:100%;height:auto;">

How to read the waterfall:

- Each row is a span: the service in bold, the operation, and its duration. A span's children are indented below it, and the bar on the right shows when the span started and how long it took, within the whole request.
- A red icon marks a span whose status is error. The panel shows times in your computer's time zone, so its start time can differ from the UTC times in Discover.
- **Span Filters** at the top narrows the rows, and **Prev** and **Next** step through the spans that match.

The first row is the load generator's `POST`, the order that it placed. The error passes up from callee to caller, so several services show the icon: the proxy, the frontend, and checkout. Scroll down to the deepest span with an error, the row **payment** `grpc.oteldemo.PaymentService/Charge`. The spans below it, such as `charge`, have no error. Click the row. Its details open below it, with **Status** `error`. Click **Span Attributes** to expand them: `grpc` has `"error_message":"Payment request failed. Invalid token. app.loyalty.level=gold"`. This is where the failure starts.

<img src="images/failed-checkout-trace.png" alt="The Trace Panel of a failed checkout: the waterfall with a red error icon on checkout's call to payment, and the details of the payment service's Charge span with Status error and its span attributes, where grpc holds the error message Payment request failed" style="max-width:100%;height:auto;">

The notebook lists the error spans of a failed checkout from at least a minute ago:

In [ ]:
failed = lab.query(f"""
SELECT trace_id
FROM otel_logs
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND body MATCH_PHRASE 'POST /api/checkout'
  AND body MATCH_PHRASE 'HTTP/1.1 500'
  AND timestamp < UTC_TIMESTAMP() - INTERVAL 1 MINUTE
ORDER BY timestamp DESC
LIMIT 1
""")
failed_trace = failed["trace_id"][0]
lab.sql(f"""
SELECT service_name, span_name, span_kind, status_message,
       events[1].name AS first_event,
       events[1].attributes['exception.message'] AS exception_message
FROM otel_traces
WHERE trace_id = '{failed_trace}'
  AND status_code = 'STATUS_CODE_ERROR'
ORDER BY service_name, span_name
""", title=f"Error spans of the trace {failed_trace}");

**Expected result**

About nine error spans in five services: `checkout`, `frontend`, `frontend-proxy`, `load-generator`, and `payment`. The services record the error in different ways. The payment service's `grpc.oteldemo.PaymentService/Charge` span has an `exception` event with the message `Payment request failed. Invalid token. app.loyalty.level=gold`, which the panel showed as the span's error message. Checkout's spans repeat that message in their `status_message`, after `failed to charge card: could not charge the card:`, and one frontend span has an `exception` event that wraps the same text. The order of the rows says nothing about time: the query sorts them by service. To tell where an error starts, look for the deepest span with an error, as in the waterfall: its children, if any, have no error.

## 6. Search the Traces page

Discover searches rows; the **Traces** page searches whole traces. Open it with **Traces** under **Doris App** in Grafana's menu, or at [http://127.0.0.1:23000/a/velodb-doris-app/traces](http://127.0.0.1:23000/a/velodb-doris-app/traces). It reads `otel_traces`.

1. Set the **Time Range** to the same absolute range as in Discover, and click **Apply time range**.
2. Leave **Service** and **Operation** at **ALL**. They select traces by their first span, which for these orders is the load generator's `POST`, not a span of checkout.
3. In **Tags**, type `http.status_code=500`. Tags are span attributes: this finds traces that have a span with the attribute `http.status_code` set to `500`, such as the proxy's span of a failed checkout.
4. Click **Find Traces**.

The chart shows each trace as a dot, by start time and duration, and the list below shows each trace with its first span (`load-generator:POST`), its trace ID, its number of spans, the services in it, and its duration. Click a trace in the list to open its waterfall.

In [ ]:
lab.sql(f"""
SELECT COUNT(DISTINCT trace_id) AS traces_with_http_500
FROM otel_traces
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND span_attributes['http.status_code'] = '500'
""", title="Traces with an HTTP 500 span in the time range");

**Expected result**

The number of traces that the page lists, about the same as the number of failed checkouts in section 4; every listed trace contains `payment` and `checkout`. A trace that reached the page after the access log search can make the counts differ by one or two. The page answers questions about requests, such as "which requests failed, and how long did they take?", while Discover answers questions about rows.

## 7. Rule out the host

Could the failures come from an overloaded machine? The DOG Collector measures the CPU of the Docker host every 15 seconds: Docker Desktop's Linux virtual machine on macOS and Windows, your computer on Linux. It writes the measurements to `otel_metrics_gauge` as the metric `system.cpu.utilization`.

Look at the rows first. In Discover, set **Table** to `otel_metrics_gauge`, keep the same time range, and in Lucene mode search for:

```text
metric_name:"system.cpu.utilization"
```

Click the arrow at the left of a row to expand it. Its `attributes` name one CPU core and one state, such as `{"cpu":"cpu3","state":"idle"}`, and `value` is the share of that core's time that it spent in that state since the previous measurement, between 0 and 1. A busy host therefore shows up as small `idle` values, and a single row says nothing about the whole host.

The share of time that the host was busy is 1 minus the average `idle` share over all cores. Several programs report a metric of this name; the DOG Collector's rows are those with the host's name in `resource_attributes` (Module 8 explains the others). Compute the busy share per minute:

In [ ]:
lab.sql("""
SELECT DATE_FORMAT(timestamp, '%H:%i') AS minute_utc,
       ROUND((1 - AVG(value)) * 100, 1) AS host_cpu_busy_percent
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND attributes['state'] = 'idle'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
GROUP BY minute_utc
ORDER BY minute_utc
""", title="Host CPU busy, per minute, over the last 15 minutes");

**Expected result**

One row per minute, usually between 5 and 15 percent busy, before and after you turned on the flag. The host has CPU to spare, and the failures did not change its load: the cause is in the application, as the payment span said. Module 8 alerts when this share stays above 90 percent.

## 8. Build an incident dashboard

A dashboard keeps the queries of an investigation in panels that refresh themselves. Its panels query Doris through Grafana's Doris data source, with SQL. They convert times correctly, so on a dashboard a relative range such as **Last 30 minutes** works in any time zone.

Creating a dashboard needs an account:

1. In Grafana, click **Sign in** at the top right. Sign in as `admin` with the password `admin`. Grafana then asks for a new password: click **Skip**.

Add the first panel, the orders per minute by result:

2. Open **Dashboards** in the menu, or [http://127.0.0.1:23000/dashboards](http://127.0.0.1:23000/dashboards). Click **New**, then **New dashboard**, then **Add visualization**.
3. In **Select data source**, click **Doris**. The panel editor opens: the panel at the top, the query editor below it, and the panel's options on the right, with **Time series** as the visualization.
4. In the query editor, switch from **Builder** to **Code**, and replace the query with:

   ```sql
   SELECT $__timeGroupAlias(timestamp, 1m),
          SUM(IF(status_code = 'STATUS_CODE_ERROR', 0, 1)) AS succeeded,
          SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed
   FROM otel_traces
   WHERE $__timeFilter(timestamp)
     AND service_name = 'checkout'
     AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
   GROUP BY 1
   ORDER BY 1
   ```

   `$__timeFilter(timestamp)` and `$__timeGroupAlias(timestamp, 1m)` are Grafana macros: Grafana replaces them with a condition for the dashboard's time range and with the start of each minute, named `time`.
5. Change **Format** from **Table** to **Time series**, and click **Run query**. Two lines appear, `succeeded` and `failed`.
6. On the right, set **Title** to `Orders per minute`. Under **Graph styles**, set **Style** to **Bars** and **Stack series** to **Normal**: each minute's bar then shows all of its orders, split into succeeded and failed.
7. Click **Back to dashboard**.

Add the second panel, the host CPU:

8. Click **Add**, then **Visualization**. The editor opens with the Doris data source.
9. Switch to **Code**, replace the query with the following, set **Format** to **Time series**, and click **Run query**:

   ```sql
   SELECT $__timeGroupAlias(timestamp, 1m),
          (1 - AVG(value)) * 100 AS busy_percent
   FROM otel_metrics_gauge
   WHERE $__timeFilter(timestamp)
     AND metric_name = 'system.cpu.utilization'
     AND attributes['state'] = 'idle'
     AND scope_name LIKE '%hostmetricsreceiver%'
     AND resource_attributes['host.name'] IS NOT NULL
   GROUP BY 1
   ORDER BY 1
   ```

10. Set **Title** to `Host CPU busy (%)` and click **Back to dashboard**.

Save the dashboard:

11. Set the dashboard's time range, at the top right, to **Last 30 minutes**.
12. Click **Save dashboard**, set **Title** to `Payment incident`, and click **Save**.

<img src="images/payment-incident-dashboard.png" alt="The Payment incident dashboard over the last 30 minutes: the panel Host CPU busy with a line around 8 percent, and the panel Orders per minute with stacked bars of succeeded and failed orders" style="max-width:100%;height:auto;">

Read the saved dashboard back from Grafana:

In [ ]:
lab.grafana_dashboard("Payment incident");

**Expected result**

Two rows, one per panel: `Orders per minute` and `Host CPU busy (%)`, both `timeseries` with the format `time_series`, and the SQL that you entered. The table's title links to the dashboard. If the cell reports that Grafana has no dashboard named `Payment incident`, save the dashboard with exactly that title.

On the dashboard, most bars since you turned on the flag have a `failed` part, while the host CPU line stays low. The dashboard stays in Grafana's volume: `lab.teardown(keep_data=True)` keeps it, and `lab.teardown()` removes it.

## 9. Exercise: the share of failed charges

The payment service records each charge request as a server span named `grpc.oteldemo.PaymentService/Charge`. Write a query that returns, for the last 15 minutes, how many charge requests the payment service received, how many failed, and the failed share in percent.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.sql("""
SELECT COUNT(*) AS charges,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_charges,
       ROUND(100 * SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) / COUNT(*), 1) AS failed_percent
FROM otel_traces
WHERE service_name = 'payment'
  AND span_name = 'grpc.oteldemo.PaymentService/Charge'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
""", title="Charge requests of the last 15 minutes");

**Expected result**

One row. If the flag has been on for the whole 15 minutes, `failed_percent` is close to 50, the value of the flag; with a few dozen charges, anything from about 35 to 65 is normal. If you turned on the flag more recently, the share is lower, because the earlier charges all succeeded.

## You are done when

- you found the failed checkouts in Discover, with Lucene and with SQL, and the counts match the notebook's;
- you opened the trace of a failed checkout and found the payment service's error, `Payment request failed. Invalid token. app.loyalty.level=gold`;
- the host CPU stays far below 90 percent;
- `lab.grafana_dashboard("Payment incident")` lists the two panels.

## Next

Open [Quiz 3: Visualizing Observability Data](quiz03_observability_visualization.ipynb). This is the last Lab of Level 1; Level 2 starts with [Module 4: Table Models for Observability Data](../../level2/module04-table-model/course.md).